In [69]:
# Imports
import json
import concurrent.futures
import re
import os
from textwrap import dedent
from statistics import mean
from dotenv import load_dotenv
from anthropic import Anthropic

In [70]:
# Client Initialization and helper functions
from anthropic.types import Message, ToolParam


load_dotenv()
API_KEY=os.getenv("CLAUDE_API_KEY")

client = Anthropic(api_key=API_KEY)
model = "claude-haiku-4-5"


def add_user_message(messages, message):
    user_message = {"role": "user", "content": message.content if isinstance(message, Message) else message}
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {"role": "assistant", "content": message.content if isinstance(message, Message) else message}
    messages.append(assistant_message)

def text_from_message(message):
    return "\n".join(
        [block.text for block in message.content if block.type == "text"]
    )

def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences
    }

    if system:
        params["system"] = system
    if tools:
        params["tools"] = tools
    return client.messages.stream(**params)

def save_article(input):
    print(input)
    return "article saved successfully!"

save_article_schema = ToolParam({
    "name": "save_article",
    "description": "Saves a scholarly article",
    "eager_input_streaming": True,
    "input_schema": {
        "type": "object",
        "properties": {
            "abstract": {
                "type": "string",
                "description": "The article summary text"
            },
            "meta": {
                "type": "string",
                "description": "Detailed description of the article."
            }
        }
    }
})

def run_tools(message):
    tool_requests = [m for m in message.content if m.type == 'tool_use']
    tool_results = []
    for r in tool_requests:
        if r.name == 'save_article':
            ret = save_article(r.input)
            tool_results.append({
                    'tool_use_id': r.id,
                    'type': 'tool_result',
                    'content': ret,
                    'is_error': False
                })
    return tool_results

def run_conversation(messages):
    while True:
        with chat(messages=messages, tools=[save_article_schema]) as stream:
            for chunk in stream:
                if chunk.type == "text":
                    print(chunk.text, end="")

                if chunk.type == "content_block_start":
                    if chunk.content_block.type == "tool_use":
                        print(f'\n>>> Tool Call: "{chunk.content_block.name}"')

                if chunk.type == "input_json" and chunk.partial_json:
                    print(f'\n>>> Input JSON: {chunk.partial_json}', end="")

                if chunk.type == "content_block_stop":
                    print("\n")
            response = stream.get_final_message()
            add_assistant_message(messages, response)
            
            if response.stop_reason != 'tool_use':
                break

            output = run_tools(response)
            add_user_message(messages, output)


In [71]:

prompt = f"""
Generate and save a scholarly article.

Guidelines:

- abstract is only 255 characters
- meta is 1000 characters
"""

messages = []
add_user_message(messages, prompt)
run_conversation(messages)
messages


>>> Tool Call: "save_article"

>>> Input JSON: {"abstract": "This study examines the relationship between digital literacy and academic
>>> Input JSON:  performance in higher education. Using a mixed-methods approach with 450 students,
>>> Input JSON:  we found significant positive correlations between specific digital skills and course outcomes,
>>> Input JSON:  particularly in collaborative online learning environments.
>>> Input JSON: ", "meta": "This comprehensive scholarly article investigates how digital literacy compet
>>> Input JSON: encies influence academic achievement among undergraduate and graduate students in contemporary higher education settings. The research employs both quantitative statistical analysis and qualitative interview
>>> Input JSON:  data collected from four diverse institutions over an 
>>> Input JSON: 18-month period. Key findings reveal that students with advanced proficiency in digital collaboration tools, information
>>> Input JSON:  evaluation, and 

[{'role': 'user',
  'content': '\nGenerate and save a scholarly article.\n\nGuidelines:\n\n- abstract is only 255 characters\n- meta is 1000 characters\n'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01G8a15CG1x95SAPBr5o583R', caller=DirectCaller(type='direct'), input={'abstract': 'This study examines the relationship between digital literacy and academic performance in higher education. Using a mixed-methods approach with 450 students, we found significant positive correlations between specific digital skills and course outcomes, particularly in collaborative online learning environments.', 'meta': 'This comprehensive scholarly article investigates how digital literacy competencies influence academic achievement among undergraduate and graduate students in contemporary higher education settings. The research employs both quantitative statistical analysis and qualitative interview data collected from four diverse institutions over an 18-month period. Key findings revea